# Owl
## Stochastic Spending Optimization — Prototype

This notebook demonstrates a **stochastic recourse LP** for retirement spending decisions using Owl's public API.

### The Problem

Owl's standard `solve()` is *certainty-equivalent*: it finds the optimal plan for one rate scenario at a time.
When run over many historical scenarios, each produces a different optimal spending basis $g_0^s$.
A retiree must commit to a spending level *before* knowing which scenario will unfold.

### The Approach

We find a common first-year commitment $g^*$ that maximizes expected spending subject to a penalty on scenarios
where the commitment exceeds what the scenario can sustain (shortfall):

$$\max_g \ g - \lambda \cdot \frac{1}{S} \sum_s \sigma_s \quad \text{s.t.} \quad \sigma_s \geq g - g_0^s, \ \sigma_s \geq 0$$

The scalar $\lambda$ controls the trade-off:
- $\lambda = 0$: maximize spending ignoring downside risk
- $\lambda \to \infty$: maximize worst-case spending (maximin = 1966 scenario)
- Intermediate $\lambda$: the **efficient frontier** between expected spending and security

This is strictly superior to Guyton-Klinger heuristics: it is globally optimal under the scenario set,
accounts for all of Owl's tax/Medicare/SS interactions, and makes the risk trade-off explicit.

Copyright &copy; 2025-2026 - Martin-D. Lacasse

Disclaimers: *I am not a financial planner. You make your own decisions. This program comes with no guarantee. Use at your own risk.*

In [ ]:
import sys
sys.path.insert(0, '../src')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.optimize import linprog

import owlplanner as owl

## Step 1: Run historical scenarios

We load Robin's case and run it over all valid historical start years.
Robin is 65 with a 26-year horizon (2026–2051), so start years 1928–2000 give 73 non-cycling scenarios.

For each scenario we record:
- `basis`: optimal spending in today's dollars (the spending scale $g_0^s$)
- `g_n`: nominal spending per year (shape N_n,)

In [ ]:
p = owl.readConfig('../examples/Case_robin.toml')

options = {
    'maxRothConversion': 50,
    'bequest': 50,
    'withSCLoop': True,
    'withMedicare': 'loop',
    'previousMAGIs': [71.0, 72.0],
    'solver': 'MOSEK',
}

# Valid start years: need start + N_n - 1 <= last data year (2025)
# N_n is printed after solve; last valid start year = 1999
START_YEARS = range(1928, 2000)

scenarios = []
failed = []

for year in START_YEARS:
    p.setRates('historical', year)
    p.solve('maxSpending', options=options)
    if p.caseStatus == 'solved':
        scenarios.append({
            'year': year,
            'basis': p.basis,
            'g_n': p.g_n.copy(),
        })
    else:
        failed.append(year)

S = len(scenarios)
print(f'Solved {S} scenarios ({len(failed)} failed: {failed})')

In [ ]:
# Extract arrays for downstream use
start_years = np.array([s['year'] for s in scenarios])
bases = np.array([s['basis'] for s in scenarios])          # today's dollars, shape (S,)
g_ns = np.array([s['g_n'] for s in scenarios])             # nominal dollars, shape (S, N_n)

N_n = g_ns.shape[1]
print(f'Planning horizon: {N_n} years')
print(f'Spending range across scenarios: ${bases.min():,.0f} - ${bases.max():,.0f}/year (today$)')
print(f'Worst-case start year: {start_years[bases.argmin()]} (${bases.min():,.0f})')
print(f'Best-case start year:  {start_years[bases.argmax()]} (${bases.max():,.0f})')

### Distribution of optimal spending across scenarios

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(start_years, bases / 1000, color='steelblue', alpha=0.7, width=0.8)
ax.axhline(bases.mean() / 1000, color='navy', linestyle='--', label=f'Mean ${bases.mean():,.0f}')
ax.axhline(bases.min() / 1000, color='firebrick', linestyle=':', label=f'Min ${bases.min():,.0f} ({start_years[bases.argmin()]})')
ax.set_xlabel('Historical start year')
ax.set_ylabel('Optimal spending (k$/year, today$)')
ax.set_title("Robin's optimal spending by historical start year")
ax.legend()
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}k'))
plt.tight_layout()
plt.show()

## Step 2: Stochastic recourse LP

For a given risk-aversion parameter $\lambda$, we solve:

$$\min_{g, \boldsymbol{\sigma}} \; -g + \frac{\lambda}{S} \mathbf{1}^\top \boldsymbol{\sigma}
\quad \text{s.t.} \quad
g - \sigma_s \leq g_0^s \; \forall s, \quad \sigma_s \geq 0, \quad g \geq 0$$

Variables: $x = [g, \sigma_0, \dots, \sigma_{S-1}]^\top$ — $S+1$ variables, $S$ inequality constraints.
Solved with `scipy.optimize.linprog` (trivially fast).

In [ ]:
def solve_stochastic_lp(bases, lam):
    """
    Solve the stochastic spending LP for a given lambda.

    Variables: x = [g, sigma_0, ..., sigma_{S-1}]
    Returns: (g_opt, expected_shortfall, shortfall_probability)
    """
    S = len(bases)

    # Objective: minimize -g + (lam/S) * sum(sigma_s)
    c = np.concatenate([[-1.0], np.full(S, lam / S)])

    # Inequality: g - sigma_s <= basis_s  =>  A_ub @ x <= b_ub
    A_ub = np.zeros((S, 1 + S))
    A_ub[:, 0] = 1.0
    A_ub[np.arange(S), 1 + np.arange(S)] = -1.0
    b_ub = bases

    # Bounds: 0 <= g <= max(bases) (cap prevents unboundedness at lambda=0),
    #         sigma_s >= 0
    bounds = [(0, float(bases.max()))] + [(0, None)] * S

    result = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')

    if result.status != 0:
        raise RuntimeError(f'LP failed: {result.message}')

    g_opt = result.x[0]
    sigmas = result.x[1:]
    expected_shortfall = sigmas.mean()
    shortfall_prob = (sigmas > 1.0).mean()   # fraction of scenarios with > $1 shortfall

    return g_opt, expected_shortfall, shortfall_prob


def g_for_success_rate(target_success_rate, lambdas, frontier_g, frontier_prob):
    """
    Return (g_opt, lam) for the least conservative lambda that achieves
    the target success rate (fraction of scenarios with no shortfall).
    """
    target_shortfall_prob = 1.0 - target_success_rate
    # frontier_prob is non-increasing; find first index where it crosses the target
    candidates = np.where(frontier_prob <= target_shortfall_prob)[0]
    if len(candidates) == 0:
        # maximin still doesn't achieve target — return maximin
        return frontier_g[-1], lambdas[-1]
    idx = candidates[0]
    return frontier_g[idx], lambdas[idx]

## Step 3: Efficient frontier

Sweep $\lambda$ from 0 (risk-neutral) to a large value (maximin) and record the Pareto frontier.
The user-facing parameter is the **success rate** — the fraction of historical scenarios in which
the spending commitment $g^*$ can be fully sustained. Lambda is an internal parameter only.

In [ ]:
lambdas = np.concatenate([[0], np.logspace(-1, 3, 60)])

frontier_g = []
frontier_shortfall = []
frontier_prob = []

for lam in lambdas:
    g_opt, exp_sf, prob_sf = solve_stochastic_lp(bases, lam)
    frontier_g.append(g_opt)
    frontier_shortfall.append(exp_sf)
    frontier_prob.append(prob_sf)

frontier_g = np.array(frontier_g)
frontier_shortfall = np.array(frontier_shortfall)
frontier_prob = np.array(frontier_prob)

# Verify endpoints
print(f'lambda=0   (risk-neutral): g = ${frontier_g[0]:,.0f}/yr  (expected: ${bases.max():,.0f})')
print(f'lambda=max (maximin):      g = ${frontier_g[-1]:,.0f}/yr  (expected: ${bases.min():,.0f})')

In [ ]:
SUCCESS_RATES = [0.70, 0.85, 0.95]
SR_COLORS = ['steelblue', 'darkorange', 'firebrick']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: spending vs. shortfall probability (user-facing axis)
ax = axes[0]
ax.plot(frontier_prob * 100, frontier_g / 1000, color='gray', linewidth=1.5)

for sr, color in zip(SUCCESS_RATES, SR_COLORS):
    g_opt, lam = g_for_success_rate(sr, lambdas, frontier_g, frontier_prob)
    shortfall_pct = (1 - sr) * 100
    ax.scatter([shortfall_pct], [g_opt / 1000], color=color, zorder=5, s=60)
    ax.annotate(f'{sr*100:.0f}% success\n${g_opt:,.0f}/yr',
                xy=(shortfall_pct, g_opt / 1000),
                xytext=(6, 4), textcoords='offset points', fontsize=8, color=color)

ax.set_xlabel('Shortfall probability (% of scenarios)')
ax.set_ylabel('Committed spending (k$/year, today$)')
ax.set_title('Efficient frontier — pick a success rate')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}k'))
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
ax.grid(True, alpha=0.3)

# Right: spending vs. expected shortfall magnitude
ax = axes[1]
ax.plot(frontier_shortfall / 1000, frontier_g / 1000, color='gray', linewidth=1.5)

for sr, color in zip(SUCCESS_RATES, SR_COLORS):
    g_opt, lam = g_for_success_rate(sr, lambdas, frontier_g, frontier_prob)
    g_sf, _, _ = solve_stochastic_lp(bases, lam)
    _, exp_sf, _ = solve_stochastic_lp(bases, lam)
    ax.scatter([exp_sf / 1000], [g_opt / 1000], color=color, zorder=5, s=60,
               label=f'{sr*100:.0f}% success')

ax.set_xlabel('Expected shortfall (k$/year, today$)')
ax.set_ylabel('Committed spending (k$/year, today$)')
ax.set_title('Efficient frontier — expected cost of security')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}k'))
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}k'))
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.suptitle("Robin's stochastic spending efficient frontier", fontsize=11)
plt.tight_layout()
plt.show()

## Step 4: Scenario outcome chart

For a flat spending profile, the achieved spending in today's dollars is simply
$\min(g^*, g_0^s)$ — constant across years within each scenario.
The interesting question is therefore not *when* spending varies, but *which historical sequences*
cause a shortfall and by how much.

A bar chart by start year (today's dollars) answers this directly, with colors indicating
success (no shortfall) vs. shortfall.

In [ ]:
def plot_scenario_outcomes(ax, start_years, bases, g_opt, title):
    """
    Bar chart of achieved spending by historical start year (today's dollars).
    Green = no shortfall, red = shortfall below commitment.
    For a flat spending profile, achieved real spending is min(g*, basis_s) — constant
    within each scenario, so the bar height fully captures the outcome.
    """
    achieved = np.minimum(g_opt, bases)
    success = achieved >= g_opt - 1.0    # $1 tolerance for floating point

    ax.bar(start_years[success], achieved[success] / 1000,
           color='mediumseagreen', alpha=0.8, width=0.9, label='No shortfall')
    ax.bar(start_years[~success], achieved[~success] / 1000,
           color='tomato', alpha=0.8, width=0.9, label='Shortfall')
    ax.axhline(g_opt / 1000, color='black', linestyle='--', linewidth=1.5,
               label=f'Commitment ${g_opt:,.0f}')

    ax.set_xlabel('Historical start year')
    ax.set_ylabel('Achieved spending (k$/yr, today$)')
    ax.set_title(title, fontsize=10)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}k'))
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3, axis='y')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for ax, sr, color in zip(axes, SUCCESS_RATES, SR_COLORS):
    g_opt, lam = g_for_success_rate(sr, lambdas, frontier_g, frontier_prob)
    _, _, prob_sf = solve_stochastic_lp(bases, lam)
    actual_success = 1.0 - prob_sf
    plot_scenario_outcomes(
        ax, start_years, bases, g_opt,
        f'Target: ≥{sr*100:.0f}% success\n'
        f'g* = ${g_opt:,.0f}/yr  '
        f'({actual_success*100:.0f}% success / {prob_sf*100:.0f}% shortfall)',
    )

plt.suptitle("Robin's scenario outcomes by historical start year (today's dollars)", fontsize=12)
plt.tight_layout()
plt.show()

## Step 5: Worst-case scenario breakdown

In [ ]:
worst_idx = np.argsort(bases)[:5]
print('Worst-case scenarios:')
print(f'{"Start year":>12}  {"Optimal basis":>14}  {"% below mean":>14}')
for i in worst_idx:
    pct = (bases[i] - bases.mean()) / bases.mean() * 100
    print(f'{start_years[i]:>12}  ${bases[i]:>12,.0f}  {pct:>13.1f}%')

In [ ]:
# Interactive: choose your desired success rate
my_success_rate = 0.90   # sustain spending in 90% of historical scenarios

g_opt, lam = g_for_success_rate(my_success_rate, lambdas, frontier_g, frontier_prob)
_, exp_sf, prob_sf = solve_stochastic_lp(bases, lam)

print(f'Target success rate:        {my_success_rate*100:.0f}%')
print(f'Optimal committed spending: ${g_opt:,.0f}/year (today$)')
print(f'Expected shortfall:         ${exp_sf:,.0f}/year')
print(f'Actual shortfall rate:      {prob_sf*100:.1f}% of scenarios')
print(f'Spending vs. risk-neutral:  -${bases.max() - g_opt:,.0f}/year ({(bases.max() - g_opt)/bases.max()*100:.1f}% reduction)')
print(f'Spending vs. maximin:       +${g_opt - bases.min():,.0f}/year ({(g_opt - bases.min())/bases.min()*100:.1f}% premium)')
print(f'(Implied lambda:            {lam:.1f})')

## Interpretation

### What this shows

The **efficient frontier** reveals the trade-off between spending ambition and downside risk across all
historical market sequences. The retiree can choose any point on this frontier by selecting $\lambda$.

The **spending fan** shows the distribution of actual year-by-year nominal spending outcomes under each
commitment level. As $\lambda$ increases (moving toward maximin), the fan narrows — more security, lower
expected spending. The dashed line shows the common commitment $g^*$; scenarios above sustain it fully,
scenarios below are scaled down.

### Why this is better than Guyton-Klinger

| | Guyton-Klinger | This LP |
|---|---|---|
| Optimality | Heuristic rules | Globally optimal under scenario set |
| Risk trade-off | Fixed rules (10% cut, etc.) | Explicit $\lambda$ parameter |
| Tax interactions | Not modeled | Full Owl machinery per scenario |
| SS / Medicare / IRMAA | Not modeled | Embedded in each scenario solve |
| Efficient frontier | Not available | Directly computed |

### Next steps for full integration into Owl

This prototype uses a simplified recourse model (spending scales proportionally to scenario surplus).
A full implementation inside `owlplanner` would:
1. Build $S$ coupled LP scenario blocks with **non-anticipativity constraints** on early-year decisions
2. Allow scenario-conditional Roth conversion and withdrawal adjustments — not just spending scaling
3. Fix binary decisions (IRMAA brackets, SS taxability, Roth exclusion) from a deterministic Stage-1 solve
4. Expose $\lambda$ as a user parameter with an efficient-frontier plot in the UI

Even this simplified version provides a principled, LP-optimal spending commitment — far beyond heuristic guardrails.